In [20]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf

from google.colab import drive
from pathlib import Path
from tqdm.auto import tqdm

In [3]:
drive_path = Path("/content/drive/MyDrive")

if not drive_path.exists():
    print("Google Drive is not mounted.")
    drive.mount("/content/drive")

project_dir = drive_path / "Nexar_Project"

print("\nGoogle Drive connected:", drive_path.exists())
print("Project directory exists:", project_dir.exists())

if project_dir.exists():
    print("\nProject folders:")

    for item in sorted(project_dir.iterdir()):
        if item.is_dir():
            print(f"[DIR]  {item.name}")
        else:
            print(f"[FILE] {item.name}")

    for folder_name in ["positive", "negative"]:

        folder = project_dir / "Dataset_2FPS" / folder_name

        print(f"\nChecking: {folder}")
        print("Directory exists:", folder.exists())

        if folder.exists():
            files = list(folder.glob("*.npz"))
            print("NPZ files found:", len(files))

else:
    print("\nNexar_Project was not found in MyDrive.")

Google Drive is not mounted.
Mounted at /content/drive

Google Drive connected: True
Project directory exists: True

Project folders:
[DIR]  Dataset_2FPS
[DIR]  Dataset_3FPS
[DIR]  Logs
[DIR]  Manifests
[DIR]  Metadata

Checking: /content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive
Directory exists: True
NPZ files found: 750

Checking: /content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative
Directory exists: True
NPZ files found: 751


In [ ]:
positive_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
)

negative_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"
)

positive_files = sorted(positive_dir.glob("*.npz"))
negative_files = sorted(negative_dir.glob("*.npz"))

print("Positive files:", len(positive_files))
print("Negative files:", len(negative_files))
print("Total files:", len(positive_files) + len(negative_files))

def inspect_npz(file_path, label):

    print(f"\n{'=' * 50}")
    print(f"{label} SAMPLE")
    print(f"File: {file_path.name}")

    try:
        with np.load(file_path, allow_pickle=False) as data:
            print("Keys:", data.files)

            for key in data.files:
                array = data[key]
                print(f"\nKey: {key}")
                print("Shape:", array.shape)
                print("Data type:", array.dtype)
                print("Size:", f"{array.nbytes / (1024**2):.2f} MB")

    except Exception as error:
        print("Error:", error)

if positive_files:
    inspect_npz(positive_files[0], "POSITIVE")

if negative_files:
    inspect_npz(negative_files[0], "NEGATIVE")

Positive files: 750
Negative files: 751
Total files: 1501

POSITIVE SAMPLE
File: 00000.npz
Keys: ['frames', 'timestamps']

Key: frames
Shape: (42, 224, 224, 3)
Data type: uint8
Size: 6.03 MB

Key: timestamps
Shape: (42,)
Data type: float32
Size: 0.00 MB

NEGATIVE SAMPLE
File: 01040.npz
Keys: ['frames', 'timestamps']

Key: frames
Shape: (79, 224, 224, 3)
Data type: uint8
Size: 11.34 MB

Key: timestamps
Shape: (79,)
Data type: float32
Size: 0.00 MB


In [ ]:
positive_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
)

negative_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"
)


def analyze_video_lengths(files, label):
    frame_counts = []
    durations = []
    timestamp_issues = []
    unreadable_files = []

    for file_path in files:

        try:
            with np.load(file_path, allow_pickle=False) as data:
                timestamps = data["timestamps"]

            frame_counts.append(len(timestamps))

            if len(timestamps) > 0:
                durations.append(
                    float(timestamps[-1] - timestamps[0])
                )

            if len(timestamps) > 1:
                if not np.all(np.diff(timestamps) > 0):
                    timestamp_issues.append(file_path.name)

        except Exception as error:
            unreadable_files.append((file_path.name, str(error)))

    if frame_counts:
        print(f"\n{'=' * 45}")
        print(f"{label} VIDEO STATISTICS")
        print(f"{'=' * 45}")

        print("Videos analyzed:", len(frame_counts))
        print("Minimum frames:", min(frame_counts))
        print("Maximum frames:", max(frame_counts))
        print("Mean frames:", round(np.mean(frame_counts), 2))
        print("Median frames:", round(np.median(frame_counts), 2))

        print("\nFrame count percentiles:")
        for p in [25, 50, 75, 90, 95, 99]:
            print(f"{p}th percentile:", round(np.percentile(frame_counts, p), 2))

        print("\nAverage timestamp duration:",
              round(np.mean(durations), 2), "seconds")

        print("Timestamp ordering issues:", len(timestamp_issues))
        print("Unreadable files:", len(unreadable_files))

        if timestamp_issues:
            print("Examples:", timestamp_issues[:5])

        if unreadable_files:
            print("Unreadable examples:", unreadable_files[:5])

    return frame_counts

positive_files = sorted(positive_dir.glob("*.npz"))
negative_files = sorted(negative_dir.glob("*.npz"))

positive_counts = analyze_video_lengths(positive_files, "POSITIVE")
negative_counts = analyze_video_lengths(negative_files, "NEGATIVE")

print("\nCLASS COMPARISON")
print("Positive videos:", len(positive_counts))
print("Negative videos:", len(negative_counts))


POSITIVE VIDEO STATISTICS
Videos analyzed: 750
Minimum frames: 7
Maximum frames: 114
Mean frames: 38.69
Median frames: 40.0

Frame count percentiles:
25th percentile: 39.0
50th percentile: 40.0
75th percentile: 41.0
90th percentile: 42.0
95th percentile: 43.0
99th percentile: 46.0

Average timestamp duration: 18.85 seconds
Timestamp ordering issues: 0
Unreadable files: 0

NEGATIVE VIDEO STATISTICS
Videos analyzed: 750
Minimum frames: 30
Maximum frames: 120
Mean frames: 74.64
Median frames: 81.0

Frame count percentiles:
25th percentile: 79.0
50th percentile: 81.0
75th percentile: 81.0
90th percentile: 82.0
95th percentile: 82.55
99th percentile: 91.0

Average timestamp duration: 36.82 seconds
Timestamp ordering issues: 0
Unreadable files: 1
Unreadable examples: [('01214_0ff59ce23a0041038f893b6b54377f5d.uploading.npz', 'File is not a zip file')]

CLASS COMPARISON
Positive videos: 750
Negative videos: 750


<div dir="rtl" style="
    direction: rtl;
    text-align: justify;
    font-family: Vazirmatn, Tahoma, sans-serif;
    line-height: 2.1;
    color: #263238;
    max-width: 100%;
">

<h1 style="text-align:center; color:#1565C0;">
گزارش تحلیل آماری و تعیین تعداد فریم ورودی مدل
</h1>

<h3 style="text-align:center; color:#607D8B;">
پروژه تشخیص تصادف Nexar
</h3>

<hr>

<h2 style="color:#1565C0;">۱. هدف تحلیل</h2>

<p>
هدف این بررسی، تعیین تعداد مناسب فریم برای ورودی معماری
<strong>ConvNeXt-Tiny + BiLSTM</strong>
است. برای این تصمیم، ابتدا توزیع آماری تعداد فریم‌های ویدئوهای مثبت و منفی بررسی شده و سپس بر اساس نتایج، طول توالی ورودی مدل انتخاب می‌شود.
</p>

<p>
در این پروژه، اطلاعات زمانی اهمیت زیادی دارند؛ زیرا مدل باید بتواند تغییرات متوالی صحنه، نزدیک‌شدن خودروها و الگوهای منتهی به تصادف را یاد بگیرد.
</p>

<h2 style="color:#1565C0;">۲. مشخصات دیتاست</h2>

<table style="width:100%; border-collapse:collapse; text-align:center;" border="1" cellpadding="8">
<tr style="background:#E3F2FD;">
<th>مشخصه</th><th>مقدار</th>
</tr>
<tr><td>تعداد کل ویدئوها</td><td>۱۵۰۰</td></tr>
<tr><td>ویدئوهای مثبت (Collision)</td><td>۷۵۰</td></tr>
<tr><td>ویدئوهای منفی (No Collision)</td><td>۷۵۰</td></tr>
<tr><td>نرخ نمونه‌برداری</td><td>۲ FPS</td></tr>
<tr><td>ابعاد فریم</td><td>۲۲۴ × ۲۲۴</td></tr>
<tr><td>فضای رنگی</td><td>RGB</td></tr>
<tr><td>نوع داده</td><td>uint8</td></tr>
<tr><td>فرمت ذخیره‌سازی</td><td>Compressed NPZ</td></tr>
</table>

<h2 style="color:#1565C0;">۳. بررسی آماری تعداد فریم‌ها</h2>

<h3 style="color:#00897B;">۳.۱. ویدئوهای مثبت (Positive Samples)</h3>

<p>تعداد نمونه‌های بررسی‌شده: <strong>۷۵۰ ویدئو</strong></p>

<table style="width:100%; border-collapse:collapse; text-align:center;" border="1" cellpadding="8">
<tr style="background:#E0F2F1;">
<th>معیار آماری</th><th>مقدار</th>
</tr>
<tr><td>Minimum</td><td>۷</td></tr>
<tr><td>Maximum</td><td>۱۱۴</td></tr>
<tr><td>Mean</td><td>۳۸٫۶۹</td></tr>
<tr><td>Median</td><td>۴۰</td></tr>
<tr><td>25th Percentile</td><td>۳۹</td></tr>
<tr><td>75th Percentile</td><td>۴۱</td></tr>
<tr><td>90th Percentile</td><td>۴۲</td></tr>
<tr><td>95th Percentile</td><td>۴۳</td></tr>
<tr><td>99th Percentile</td><td>۴۶</td></tr>
</table>

<h3 style="color:#00897B;">۳.۲. ویدئوهای منفی (Negative Samples)</h3>

<p>تعداد نمونه‌های بررسی‌شده: <strong>۷۵۰ ویدئو</strong></p>

<table style="width:100%; border-collapse:collapse; text-align:center;" border="1" cellpadding="8">
<tr style="background:#E0F2F1;">
<th>معیار آماری</th><th>مقدار</th>
</tr>
<tr><td>Minimum</td><td>۳۰</td></tr>
<tr><td>Maximum</td><td>۱۲۰</td></tr>
<tr><td>Mean</td><td>۷۴٫۶۴</td></tr>
<tr><td>Median</td><td>۸۱</td></tr>
<tr><td>25th Percentile</td><td>۷۹</td></tr>
<tr><td>75th Percentile</td><td>۸۱</td></tr>
<tr><td>90th Percentile</td><td>۸۲</td></tr>
<tr><td>95th Percentile</td><td>۸۲٫۵۵</td></tr>
<tr><td>99th Percentile</td><td>۹۱</td></tr>
</table>

<h2 style="color:#1565C0;">۴. تحلیل نتایج آماری</h2>

<p>
با بررسی نتایج مشخص می‌شود که توزیع تعداد فریم‌ها در دو کلاس یکسان نیست.
</p>

<ul>
<li>میانگین تعداد فریم در ویدئوهای مثبت حدود ۳۹ فریم است.</li>
<li>میانه تعداد فریم در ویدئوهای مثبت برابر با ۴۰ است.</li>
<li>صدک ۹۹ ویدئوهای مثبت برابر با ۴۶ فریم است.</li>
<li>میانگین تعداد فریم در ویدئوهای منفی حدود ۷۵ فریم است.</li>
<li>میانه تعداد فریم در ویدئوهای منفی برابر با ۸۱ است.</li>
</ul>

<p>
این تفاوت نشان می‌دهد که ویدئوهای منفی معمولاً طولانی‌تر از ویدئوهای مثبت هستند. بنابراین انتخاب تعداد فریم باید به شکلی باشد که اطلاعات ویدئوهای مثبت تا حد امکان حفظ شود و در عین حال از ویدئوهای منفی نیز نمونه‌های مناسبی از کل بازه زمانی دریافت کنیم.
</p>

<h2 style="color:#1565C0;">۵. توجیه انتخاب ۴۸ فریم</h2>

<p>
با توجه به نتایج تحلیل آماری، تعداد
<strong style="color:#00897B;">۴۸ فریم</strong>
به‌عنوان طول ثابت توالی ورودی مدل انتخاب می‌شود.
</p>

<h3 style="color:#00897B;">دلیل اول: پوشش تقریباً کامل ویدئوهای مثبت</h3>

<p>
صدک ۹۹ تعداد فریم‌های ویدئوهای مثبت برابر با ۴۶ است. بنابراین تقریباً ۹۹ درصد ویدئوهای مثبت حداکثر ۴۶ فریم دارند.
</p>

<p>
با انتخاب ۴۸ فریم، ظرفیت کافی برای حفظ تمام فریم‌های واقعی تقریباً تمام ویدئوهای مثبت فراهم می‌شود. این موضوع مهم است؛ زیرا نمی‌خواهیم اطلاعات زمانی مربوط به نزدیک‌شدن به لحظه تصادف را بدون ضرورت حذف کنیم.
</p>

<h3 style="color:#00897B;">دلیل دوم: نمونه‌برداری مناسب از ویدئوهای منفی</h3>

<p>
ویدئوهای منفی به‌طور میانگین حدود ۷۵ فریم دارند. برای این ویدئوها، ۴۸ فریم با روش
<strong>Uniform Sampling</strong>
از سراسر ویدئو انتخاب می‌شود.
</p>

<p>
در این روش، فریم‌ها به‌صورت منظم از بخش‌های مختلف ویدئو انتخاب می‌شوند تا مدل فقط با ابتدای ویدئو مواجه نشود و بتواند اطلاعات کل بازه را دریافت کند.
</p>

<h3 style="color:#00897B;">دلیل سوم: تعادل میان اطلاعات و هزینه محاسباتی</h3>

<p>
انتخاب ۱۶ فریم ممکن است باعث حذف بخشی از اطلاعات زمانی شود. در مقابل، انتخاب ۸۰ یا ۱۲۰ فریم هزینه پردازش و مصرف حافظه را افزایش می‌دهد.
</p>

<p>
انتخاب ۴۸ فریم یک نقطه شروع منطقی است که اطلاعات زمانی بیشتری نسبت به ۱۶ فریم فراهم می‌کند و در عین حال طول توالی را در محدوده قابل مدیریت برای GPU مدل
<strong>NVIDIA Tesla T4</strong>
در Google Colab نگه می‌دارد.
</p>

<h2 style="color:#1565C0;">۶. نحوه آماده‌سازی توالی ۴۸ فریمی</h2>

<h3 style="color:#00897B;">ویدئوهای مثبت</h3>

<ul>
<li>اگر تعداد فریم‌ها حداکثر ۴۸ باشد، تمام فریم‌های واقعی حفظ می‌شوند.</li>
<li>اگر تعداد فریم‌ها بیشتر از ۴۸ باشد، ۴۸ فریم به‌صورت منظم از توالی انتخاب می‌شوند.</li>
<li>اگر تعداد فریم‌ها کمتر از ۴۸ باشد، از Padding برای تکمیل طول توالی استفاده می‌شود.</li>
</ul>

<p>
نکته بسیار مهم: در نمونه‌های مثبت، هیچ فریمی از زمان وقوع تصادف یا پس از آن نباید وارد مدل شود. این محدودیت برای جلوگیری از نشت اطلاعات آینده (Future Information Leakage) ضروری است.
</p>

<h3 style="color:#00897B;">ویدئوهای منفی</h3>

<ul>
<li>اگر تعداد فریم‌ها بیشتر از ۴۸ باشد، ۴۸ فریم با Uniform Sampling از کل ویدئو انتخاب می‌شوند.</li>
<li>اگر تعداد فریم‌ها برابر با ۴۸ باشد، تمام فریم‌ها استفاده می‌شوند.</li>
<li>اگر تعداد فریم‌ها کمتر از ۴۸ باشد، از Padding استفاده می‌شود.</li>
</ul>

<h2 style="color:#1565C0;">۷. Padding و Masking</h2>

<p>
برخی ویدئوهای مثبت فقط ۷ فریم دارند. برای تبدیل طول این نمونه‌ها به ۴۸، از Padding استفاده می‌کنیم.
</p>

<table style="width:100%; border-collapse:collapse; text-align:center;" border="1" cellpadding="8">
<tr style="background:#FFF3E0;">
<th>نوع داده</th><th>تعداد</th>
</tr>
<tr><td>فریم واقعی</td><td>۷</td></tr>
<tr><td>جایگاه تکمیلی</td><td>۴۱</td></tr>
<tr><td>طول نهایی توالی</td><td>۴۸</td></tr>
</table>

<p>
اما Padding به‌تنهایی کافی نیست. باید با استفاده از
<strong>Masking</strong>
به بخش BiLSTM مشخص کنیم کدام فریم‌ها واقعی و کدام جایگاه‌ها تکمیلی هستند؛ در غیر این صورت ممکن است مدل از داده‌های مصنوعی Padding نیز یاد بگیرد.
</p>

<h2 style="color:#1565C0;">۸. چرا فعلاً از ۱۲۰ فریم استفاده نمی‌کنیم؟</h2>

<ul>
<li>بیشتر ویدئوهای مثبت حدود ۴۰ فریم دارند.</li>
<li>توالی ۱۲۰ فریمی برای بسیاری از ویدئوهای مثبت با Padding گسترده همراه می‌شود.</li>
<li>پردازش فریم‌های بیشتر هزینه محاسباتی و مصرف حافظه را افزایش می‌دهد.</li>
<li>افزایش تعداد فریم‌ها الزاماً باعث افزایش دقت مدل نمی‌شود.</li>
<li>ابتدا باید یک مدل پایه قابل ارزیابی بسازیم و سپس اثر طول توالی بیشتر را آزمایش کنیم.</li>
</ul>

<h2 style="color:#1565C0;">۹. تنظیمات اولیه مدل</h2>

<table style="width:100%; border-collapse:collapse; text-align:center;" border="1" cellpadding="8">
<tr style="background:#E3F2FD;">
<th>پارامتر</th><th>مقدار</th>
</tr>
<tr><td>Architecture</td><td>ConvNeXt-Tiny + BiLSTM</td></tr>
<tr><td>Sequence Length</td><td><strong>48 Frames</strong></td></tr>
<tr><td>Image Size</td><td>224 × 224</td></tr>
<tr><td>Color Space</td><td>RGB</td></tr>
<tr><td>Sampling Rate</td><td>2 FPS</td></tr>
<tr><td>Batch Size</td><td>1 یا 2، پس از تست حافظه</td></tr>
<tr><td>Precision</td><td>mixed_float16</td></tr>
<tr><td>Framework</td><td>TensorFlow / Keras</td></tr>
<tr><td>GPU</td><td>NVIDIA Tesla T4</td></tr>
</table>

<h2 style="color:#1565C0;">۱۰. آزمایش‌های مقایسه‌ای آینده</h2>

<p>
پس از آموزش مدل پایه با ۴۸ فریم، می‌توانیم اثر طول توالی را با آزمایش‌های کنترل‌شده بررسی کنیم.
</p>

<table style="width:100%; border-collapse:collapse; text-align:center;" border="1" cellpadding="8">
<tr style="background:#E8EAF6;">
<th>آزمایش</th><th>تعداد فریم</th><th>هدف</th>
</tr>
<tr><td>Baseline</td><td>۴۸</td><td>مدل پایه</td></tr>
<tr><td>Experiment 2</td><td>۶۴</td><td>بررسی توالی طولانی‌تر</td></tr>
<tr><td>Experiment 3</td><td>۸۰</td><td>بررسی پوشش بیشتر ویدئوهای منفی</td></tr>
</table>

<p>
در این مقایسه‌ها باید سایر شرایط، مانند تقسیم داده، معماری، روش آموزش و تنظیمات بهینه‌ساز تا حد امکان ثابت بمانند.
</p>

<p>معیارهای ارزیابی:</p>

<ul>
<li>Accuracy</li>
<li>Precision</li>
<li>Recall</li>
<li>F1-score</li>
<li>Confusion Matrix</li>
<li>Training Time</li>
<li>GPU Memory Usage</li>
</ul>

<p>
در مسئله تشخیص تصادف، Recall کلاس مثبت اهمیت ویژه‌ای دارد؛ زیرا Recall پایین به معنی از دست رفتن بخشی از تصادف‌های واقعی است. در کنار آن، Precision نیز باید بررسی شود تا تعداد هشدارهای اشتباه بیش از حد افزایش پیدا نکند.
</p>

<h2 style="color:#1565C0;">۱۱. نتیجه‌گیری</h2>

<p>
بر اساس تحلیل آماری ۱۵۰۰ ویدئوی دیتاست Nexar، طول ورودی
<strong style="color:#00897B;">۴۸ فریم</strong>
برای مرحله اول آموزش مدل انتخاب می‌شود.
</p>

<p>
مهم‌ترین دلیل این انتخاب آن است که صدک ۹۹ تعداد فریم‌های ویدئوهای مثبت برابر با ۴۶ است؛ بنابراین طول ۴۸ فریم امکان حفظ تقریباً تمام فریم‌های واقعی این گروه را فراهم می‌کند. همچنین برای ویدئوهای منفی می‌توان ۴۸ فریم را به‌صورت یکنواخت از کل ویدئو انتخاب کرد.
</p>

<p>
این انتخاب بین حفظ اطلاعات زمانی و محدودیت منابع محاسباتی GPU تعادل برقرار می‌کند. فایل‌های NPZ موجود و مشخصات پیش‌پردازش آن‌ها نیز بدون تغییر باقی خواهند ماند.
</p>

<div style="
    background:#E8F5E9;
    border-right:5px solid #2E7D32;
    padding:15px;
    border-radius:8px;
    margin-top:20px;
">
<strong style="color:#1B5E20;">تصمیم نهایی پروژه</strong>
<p style="margin:5px 0;">
استفاده از <strong>۴۸ فریم</strong> به‌عنوان طول ثابت ورودی مدل
<strong>ConvNeXt-Tiny + BiLSTM</strong>
و بررسی تجربی طول‌های ۶۴ و ۸۰ فریم در مراحل بعدی.
</p>
</div>

<p style="font-size:0.9em; color:#607D8B;">
توجه علمی: انتخاب ۴۸ فریم یک تصمیم اولیه مبتنی بر آمار دیتاست است و تضمین‌کننده افزایش دقت نیست. نتیجه نهایی باید با مقایسه عملکرد مدل‌ها روی Validation و ارزیابی نهایی روی Test تعیین شود.
</p>

</div>

In [5]:
BASE_DIR = "/content"
SPLIT_CSV_PATH = os.path.join(BASE_DIR, "video_split.csv")
assert os.path.exists(SPLIT_CSV_PATH), "Split CSV file not found!"

print("Split file loaded successfully.")

Split file loaded successfully.


In [12]:
PROJECT_DIR = "/content/drive/MyDrive/Nexar_Project"
METADATA_PATH = "/content/video_split.csv"

POSITIVE_DIR = os.path.join(
    PROJECT_DIR, "Dataset_2FPS", "positive"
)

NEGATIVE_DIR = os.path.join(
    PROJECT_DIR, "Dataset_2FPS", "negative"
)

MANIFEST_DIR = os.path.join(
    PROJECT_DIR, "Manifests"
)

os.makedirs(MANIFEST_DIR, exist_ok=True)


# Load original video split
df = pd.read_csv(METADATA_PATH)

# Normalize labels and split names
df["label"] = df["label"].astype(int)
df["split"] = df["split"].astype(str).str.lower().str.strip()

# Create NPZ filenames
df["npz_file"] = (
    df["file_name"]
    .str.replace(".mp4", ".npz", regex=False)
)

# Assign NPZ paths
df["npz_path"] = df.apply(
    lambda row: os.path.join(
        POSITIVE_DIR if row["label"] == 1 else NEGATIVE_DIR,
        row["npz_file"]
    ),
    axis=1
)

# Check file existence
df["file_exists"] = df["npz_path"].apply(os.path.isfile)

# Check dataset structure
print("DATASET DISTRIBUTION")
print(pd.crosstab(df["split"], df["label"]))

print("\nSPLIT COUNTS")
print(df["split"].value_counts())

print("\nFILE MAPPING")
print("Total videos:", len(df))
print("Existing NPZ files:", df["file_exists"].sum())
print("Missing NPZ files:", (~df["file_exists"]).sum())

# Show missing files
if (~df["file_exists"]).any():
    print("\nMISSING FILES:")
    print(
        df.loc[
            ~df["file_exists"],
            ["file_name", "label", "split", "npz_path"]
        ].head(20)
    )

# Save manifests
df.to_csv(
    os.path.join(MANIFEST_DIR, "all_manifest.csv"),
    index=False
)

train_df = df[df["split"] == "train"].copy()
validation_df = df[df["split"] == "validation"].copy()

train_df.to_csv(
    os.path.join(MANIFEST_DIR, "train_manifest.csv"),
    index=False
)

validation_df.to_csv(
    os.path.join(MANIFEST_DIR, "validation_manifest.csv"),
    index=False
)

print("\nMANIFESTS CREATED")
print("Train:", len(train_df))
print("Validation:", len(validation_df))

DATASET DISTRIBUTION
label         0    1
split               
train       600  600
validation  150  150

SPLIT COUNTS
split
train         1200
validation     300
Name: count, dtype: int64

FILE MAPPING
Total videos: 1500
Existing NPZ files: 1500
Missing NPZ files: 0

MANIFESTS CREATED
Train: 1200
Validation: 300


In [13]:
# Load manifests
train_df = pd.read_csv(
    os.path.join(MANIFEST_DIR, "train_manifest.csv")
)

validation_df = pd.read_csv(
    os.path.join(MANIFEST_DIR, "validation_manifest.csv")
)

all_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

valid_count = 0
invalid_files = []

total_frames = 0
min_frames = float("inf")
max_frames = 0

for path in tqdm(all_df["npz_path"], desc="Checking NPZ files"):
    try:
        with np.load(path, allow_pickle=False) as data:
            if "frames" not in data or "timestamps" not in data:
                raise ValueError("Missing required NPZ keys")

            frames = data["frames"]
            timestamps = data["timestamps"]

            # Check dimensions
            if frames.ndim != 4:
                raise ValueError(f"Invalid frames shape: {frames.shape}")

            if frames.shape[1:] != (224, 224, 3):
                raise ValueError(f"Unexpected frame dimensions: {frames.shape}")

            if frames.dtype != np.uint8:
                raise ValueError(f"Unexpected dtype: {frames.dtype}")

            if timestamps.ndim != 1:
                raise ValueError("Invalid timestamps dimensions")

            if len(frames) != len(timestamps):
                raise ValueError("Frames and timestamps length mismatch")

            if len(frames) == 0:
                raise ValueError("Empty video")

            if not np.isfinite(timestamps).all():
                raise ValueError("Non-finite timestamps")

            if np.any(np.diff(timestamps) < 0):
                raise ValueError("Timestamps are not ordered")

            # Update statistics
            frame_count = len(frames)

            total_frames += frame_count
            min_frames = min(min_frames, frame_count)
            max_frames = max(max_frames, frame_count)
            valid_count += 1

    except Exception as e:
        invalid_files.append({
            "path": path,
            "error": str(e)
        })


print("\nNPZ INTEGRITY REPORT")
print("-" * 40)

print("Total files:", len(all_df))
print("Valid files:", valid_count)
print("Invalid files:", len(invalid_files))

if valid_count > 0:
    print("Average frames:", round(total_frames / valid_count, 2))
    print("Minimum frames:", min_frames)
    print("Maximum frames:", max_frames)

if invalid_files:
    print("\nINVALID FILES:")
    for item in invalid_files:
        print(item["path"], "->", item["error"])

Checking NPZ files:   0%|          | 0/1500 [00:00<?, ?it/s]


NPZ INTEGRITY REPORT
----------------------------------------
Total files: 1500
Valid files: 1500
Invalid files: 0
Average frames: 56.67
Minimum frames: 7
Maximum frames: 120


In [19]:
sample = train_df.iloc[0]
sample

,0
file_name,00780.mp4
time_of_event,19.967
time_of_alert,16.9
light_conditions,Normal
weather,Cloudy
scene,Highway
time_to_accident,NaN
label,1
split,train
npz_file,00780.npz


In [26]:
def prepare_video(npz_path, sequence_length=48):
    with np.load(npz_path) as data:
        frames = data["frames"]

    total_frames = len(frames)

    output_frames = np.zeros(
        (sequence_length, 224, 224, 3),
        dtype=np.uint8
    )

    mask = np.zeros(
        (sequence_length,),
        dtype=bool
    )

    if total_frames >= sequence_length:

        indices = np.linspace(
            0,
            total_frames - 1,
            sequence_length
        ).round().astype(int)

        output_frames = frames[indices]
        mask[:] = True

    else:

        output_frames[:total_frames] = frames
        mask[:total_frames] = True

    return output_frames, mask

In [29]:
def video_generator(dataframe, shuffle=False):
    while True:
        if shuffle:
            indices = np.random.permutation(len(dataframe))
        else:
            indices = np.arange(len(dataframe))

        for idx in indices:
            row = dataframe.iloc[idx]
            frames, mask = prepare_video(
                row["npz_path"],
                sequence_length=SEQUENCE_LENGTH
            )

            label = np.float32(row["label"])

            inputs = {
                "frames": frames,
                "mask": mask
            }
            yield inputs, label

In [30]:
train_output_signature = (
    {
        "frames": tf.TensorSpec(
            shape=(48, 224, 224, 3),
            dtype=tf.uint8
        ),
        "mask": tf.TensorSpec(
            shape=(48,),
            dtype=tf.bool
        )
    },
    tf.TensorSpec(shape=(), dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: video_generator(train_df, shuffle=True),
    output_signature=train_output_signature
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: video_generator(val_df, shuffle=False),
    output_signature=train_output_signature
)

train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(1)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(1)

print("Train DataLoader created successfully.")
print("Validation DataLoader created successfully.")

Train DataLoader created successfully.
Validation DataLoader created successfully.


In [31]:
batch_inputs, batch_labels = next(iter(train_dataset))

print("Frames batch shape:", batch_inputs["frames"].shape)
print("Mask batch shape:", batch_inputs["mask"].shape)
print("Labels shape:", batch_labels.shape)

print("\nFrames dtype:", batch_inputs["frames"].dtype)
print("Mask dtype:", batch_inputs["mask"].dtype)
print("Labels:", batch_labels.numpy())

print(
    "\nValid frames:",
    tf.reduce_sum(
        tf.cast(batch_inputs["mask"][0], tf.int32)
    ).numpy()
)

Frames batch shape: (1, 48, 224, 224, 3)
Mask batch shape: (1, 48)
Labels shape: (1,)

Frames dtype: <dtype: 'uint8'>
Mask dtype: <dtype: 'bool'>
Labels: [0.]

Valid frames: 48
